# Testing the Workspace in Databricks 

Step 1: Check Your Unity Catalog Setup

First, let's verify your Unity Catalog configuration. 

In [0]:
"""
Setup notebook: Verify Unity Catalog access and create storage structure.

Purpose:
- Confirm we can access Unity Catalog
- Create Volumn for PDF/CSV storage
- Setup the foundaion for Layer 1 (Document Ingestion)
"""

# Check available catalogs
print("Available Catalogs:")
catalogs = spark.sql("SHOW CATALOGS").collect()
for cat in catalogs:
    print(f" - {cat.catalog}")

# Check schemas in YOUR catalog (my_catalog)
print("\n📂 Schemas in 'my_catalog':")
schemas = spark.sql("SHOW SCHEMAS IN my_catalog").collect()
for schema in schemas:
    print(f"  • {schema.databaseName}")

Step 2: Create a Schema for Your Project

After you run Step 1, we'll create a dedicated schema for this project:

In [0]:
"""
Create dedicated schema for agentic quality check project.

Why a separate schema?
- Organizes all project resources (volumes, tables) in one place
- Makes permissions management easier (grant access to entire schema)
- Follows best practice: one schema per application/project
"""

# Schema name following convention: <project_name>_<env>
SCHEMA_NAME = "agentic_quality_check_dev"
CATALOG = "my_catalog"  # Using YOUR catalog

# Create schema if it doesn't exist
spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA_NAME}
    COMMENT 'Schema for Agentic Quality Check RAG system - stores PDFs, CSVs, embeddings, and results'
""")

print(f"✅ Schema created: {CATALOG}.{SCHEMA_NAME}")

# Verify it was created
schemas = spark.sql(f"SHOW SCHEMAS IN {CATALOG}").collect()
print(f"\n📂 Updated schemas in '{CATALOG}':")
for schema in schemas:
    print(f"  • {schema.databaseName}")

Step 3: Create Unity Catalog Volumes

Now let's create the Volumes for storing files:

In [0]:
"""
Create Unity Catalog Volumes for file storage

Volume Structure:
- pdf_volume: Raw PDF documents uploaded by users
- csv_volume: Raw CSV data files for RAG queries
- processed_volumnet: embeddings, FAISS indexex, intermediate results

Why separate volumes?
- clear separation of concerns(raw data vs processed data)
- Easier to manage permissions (e.g. users can upload to pdfs.csv. but not modify processed)
- Aligns with architectrure Layer 1 (ingestion) and Layer 2 (processing)
"""

CATALOG = "my_catalog"
SCHEMA = "agentic_quality_check_dev"

# Create volumes
volumes_to_create = [
    ("pdfs_volume", "Storage for uploaded PDF documents"),
    ("csvs_volume", "Storage for uploaded CSV data files"),
    ("processed_volume", "Storage for embeddings and vector indexes"), 
    ("mappings_volume", "Storage for headline-to-CSV mapping metadata"), 
    ("feedback_volume", "Storage for user feedback for self-healing agent")]

for volume_name, comment in volumes_to_create:
    spark.sql(f"""
        CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{volume_name}
        COMMENT '{comment}'
    """)
    print(f"Volume created: {CATALOG}.{SCHEMA}.{volume_name}")

# Show all volumes in the schema
print(f"\n Volumes in {CATALOG}.{SCHEMA}:")

volumes = spark.sql(f"SHOW VOLUMES IN {CATALOG}.{SCHEMA}").collect()
for volume in volumes:
    print(f" {volume.volume_name}")

# Print the file paths to use 
print("n Gile paths to use in code:")
print(f" - PDFs: /Volumes/{CATALOG}/{SCHEMA}/pdfs_volume/")
print(f" - CSVs: /Volumes/{CATALOG}/{SCHEMA}/csvs_volume/")
print(f" - Processed: /Volumes/{CATALOG}/{SCHEMA}/processed_volume/")



In [0]:
print("n Gile paths to use in code:")
print(f" - PDFs: /Volumes/{CATALOG}/{SCHEMA}/pdfs_volume/")
print(f" - CSVs: /Volumes/{CATALOG}/{SCHEMA}/csvs_volume/")
print(f" - Processed: /Volumes/{CATALOG}/{SCHEMA}/processed_volume/")
print(f" - Mappings: /Volumes/{CATALOG}/{SCHEMA}/mappings_volume/")
print(f" - Feedback: /Volumes/{CATALOG}/{SCHEMA}/feedback_volume/")

# Testing the codes written in utilty - pdf_parser.py

In [0]:
%pip install pdfplumber

In [0]:
# Add src to path so Python can find moduler
import sys
sys.path.append('/Workspace/Users/gb.burcea@gmail.com/agentic_quality_check/src')

# Import directly from utils 

from utils import parse_pdf, extract_headlines

# Test 
pdf_path = "/Volumes/my_catalog/agentic_quality_check_dev/pdfs_volume/Multiplication_check.pdf"
result = parse_pdf(pdf_path)

print(f"Document: {result['filename']}")
print(f"Pages: {result['metadata']['total_pages']}")
print(f"Headlines found: {len(result['headlines'])}\n")

for h in result['headlines']:
    print(f" - [H{h['level']}] Page {h['page']}: {h['text']}")
    print(f" Paragraps: {len(h['paragraphs'])}")
    print(f" Text: {' '.join(h['paragraphs'])}")



Test get_csv_medatadata

In [0]:

import sys
sys.path.append('/Workspace/Users/gb.burcea@gmail.com/agentic_quality_check/src')

# Clear cached module to pick up changes
if 'utils' in sys.modules:
    del sys.modules['utils']
if 'utils.csv_handler' in sys.modules:
    del sys.modules['utils.csv_handler']

from utils import get_csv_metadata

result = get_csv_metadata("/Volumes/my_catalog/agentic_quality_check_dev/csvs_volume/mtc_national_pupil_characteristics_2022_to_2025.csv")

print(f"Filename: {result['filename']}")
print(f"Rows: {result['row_count']}")
print(f"Columns: {result['column_count']}")
print(f"\nFirst 3 columns:")
for col in result['columns'][:33]:
    print(f"  - {col['name']} ({col['type']}) - Role: {col['role']}")
    print(f"    Sample: {col['sample_values']}")


In [0]:
%pip install streamlit

In [0]:
%sh
cd /Workspace/Users/gb.burcea@gmail.com/agentic_quality_check
streamlit run app.py --server.port 8501

In [0]:
%sh
cd /Workspace/Users/gb.burcea@gmail.com/agentic_quality_check
nohup streamlit run app.py --server.port 8501 > streamlit.log 2>&1 &
echo "Streamlit started. Check streamlit.log for output"